# Bifurcation diagrams of the IBMF fixed points ($\sigma=0$)
Data produced on the cluster by `extract_bifurcation_summaries.py` (PBS script `run_extract_bifurcation_summaries.sh`):
one row per distinct fixed point found by `IBMF_LV_sequential_countFP`, for one graph (`seedgraph`) and all the values
of $\mu$, $N$ and $n_{\rm init}$ (including the intermediate checkpoints). Each point of the diagram is a distinct
fixed point: abscissa $\mu$, ordinate the abundance $n_0$ of species 0, colour the fraction of the runs that reached it.

In [ ]:
import glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

In [ ]:
path_data = "/run/media/david/Data/Research/Ecology/Results/IBMF/Bifurcation/ToDownload"    # downloaded csv.gz files
path_out = "/run/media/david/Data/Research/Ecology/Results/Plots_counting_fixed_points/bifurcation"
os.makedirs(path_out, exist_ok=True)


def set_style():
    plt.rcParams.update({"font.family": "serif", "mathtext.fontset": "stix", "axes.linewidth": 1.2})


def format_axis(ax, tick_fontsize=14):
    ax.tick_params(axis='both', which='minor', direction='in', top=True, right=True, labelsize=tick_fontsize, length=3)
    ax.tick_params(axis='both', which='major', direction='in', top=True, right=True, labelsize=tick_fontsize, length=5)

In [ ]:
files = sorted(glob.glob(f"{path_data}/bifurcation_*.csv.gz"))
data = pd.concat([pd.read_csv(f, dtype={'eps': str, 'sigma': str, 'mu': str}) for f in files], ignore_index=True)
data['mu_value'] = data['mu'].astype(float)
data = data.drop_duplicates(subset=['seedgraph', 'N', 'ninitcond', 'mu', 'fp'])
print(f"{len(files)} files, {len(data)} fixed points")
pairs = (data.groupby(['seedgraph', 'N', 'ninitcond'])['mu'].nunique().rename('number of mu values').reset_index())
pairs

In [ ]:
def plot_bifurcation(data, seedgraph, N, ninitcond, ax=None, cmap='viridis', markersize=14, label_fontsize=20,
                     title_fontsize=18, colorbar=True, save_to=None):
    """Distinct fixed points: n_0 vs mu, coloured by the fraction of the runs that reached each of them."""
    sub = data[(data['seedgraph'] == seedgraph) & (data['N'] == N) & (data['ninitcond'] == ninitcond)]
    sub = sub.sort_values('fraction')          # the most frequent fixed points are drawn on top
    set_style()
    new_figure = ax is None
    if new_figure:
        fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
        fig.subplots_adjust(left=0.14, bottom=0.14, right=0.97, top=0.92)
    norm = LogNorm(vmin=1.0 / ninitcond, vmax=1.0)
    sc = ax.scatter(sub['mu_value'], sub['n0'], c=sub['fraction'], cmap=cmap, norm=norm, s=markersize, lw=0)
    ax.set_xlabel(r"$\mu$", fontsize=label_fontsize)
    ax.set_ylabel(r"$n_0$", fontsize=label_fontsize)
    ax.set_title(rf"$N={N}$, $n_{{\rm init}}={ninitcond}$, graph {seedgraph}", fontsize=title_fontsize)
    format_axis(ax)
    if colorbar:
        cb = plt.colorbar(sc, ax=ax)
        cb.set_label("fraction of runs", fontsize=label_fontsize - 4)
        cb.ax.tick_params(labelsize=12)
    if new_figure:
        if save_to is not None:
            fig.savefig(save_to)
        plt.show()
    return sc

In [ ]:
# one figure for each pair (N, n_init) of each graph
for _, row in pairs.iterrows():
    g, N, ninit = int(row['seedgraph']), int(row['N']), int(row['ninitcond'])
    plot_bifurcation(data, g, N, ninit, save_to=f"{path_out}/bifurcation_sigma_0.000_seedgraph_{g}_N_{N}_ninit_{ninit}.png")

In [ ]:
# comparison of different n_init at fixed N (one row of panels per N)
for g in sorted(pairs['seedgraph'].unique()):
    for N in sorted(pairs[pairs['seedgraph'] == g]['N'].unique()):
        ninits = sorted(pairs[(pairs['seedgraph'] == g) & (pairs['N'] == N)]['ninitcond'].unique())
        set_style()
        fig, axs = plt.subplots(1, len(ninits), figsize=(4.5 * len(ninits), 4.2), dpi=120, squeeze=False)
        for ax, ninit in zip(axs[0], ninits):
            plot_bifurcation(data, g, N, ninit, ax=ax, label_fontsize=16, title_fontsize=13, markersize=8)
        fig.tight_layout()
        fig.savefig(f"{path_out}/bifurcation_sigma_0.000_seedgraph_{g}_N_{N}_all_ninit.png")
        plt.show()